In [1]:
!nvidia-smi
!nvcc --version

Tue Feb 10 21:08:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%shell
mkdir -p /content/parallel-architectures
mkdir -p /content/parallel-architectures/include/
mkdir -p /content/parallel-architectures/src/


In [ ]:
%%writefile /content/parallel-architectures/include/common.h
#ifndef COMMON_H
#define COMMON_H

#include <iostream>
#include <fstream>
#include <vector>
#include <algorithm>
#include <chrono>
#include <cstdlib>
#include <cmath>
#include <cstring>

/**
 * Common utilities for CUDA educational projects
 * Include this header in both .cpp and .cu files
 */

// Default CUDA block size (threads per block)
#define NumThPerBlock 512
// Default max number of blocks
#define MaxBlocks 80 // SMs * (maxThreadsPerSM / ThreadsPerBlock)

int gridStrideBlocks(int number_of_elements, int threads_per_block = NumThPerBlock, int max_blocks = MaxBlocks) {
    int blocks = (number_of_elements + NumThPerBlock - 1) / NumThPerBlock;
    return (blocks > MaxBlocks) ? MaxBlocks : blocks;
}

/**
 * Timer class for performance benchmarking
 * Usage:
 *   Timer timer;
 *   timer.start();
 *   // ... code to benchmark ...
 *   double elapsed_ms = timer.stop();
 */
class Timer {
private:
    std::chrono::high_resolution_clock::time_point start_time;
    
public:
    void start() {
        start_time = std::chrono::high_resolution_clock::now();
    }
    
    double stop() {
        auto end_time = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> duration = end_time - start_time;
        return duration.count();
    }
};


/**
 * Simple CSR Graph structure
 */
struct CSRRepr {
    int num_nodes;
    int num_edges;
    int* row_ptr;
    int* col_ind;
};

// Utility function to free CSR graph memory in the host
void freeCSRRepr(CSRRepr& graph) {
    delete[] graph.row_ptr;
    delete[] graph.col_ind;

    graph.num_nodes = 0;
    graph.num_edges = 0;
    graph.row_ptr = nullptr;
    graph.col_ind = nullptr;
}

int get_vertex_from_literal(int lit) {
    return (lit > 0) ? 2 * lit - 2 : 2 * (-lit) - 1;
}

/**
 * Creates a CSR graph from an edge lists.
 */
CSRRepr createCSRGraph(int num_nodes, int num_edges, const int2* edge_list) {
    CSRRepr csr;
    csr.num_nodes = num_nodes;
    csr.num_edges = num_edges;

    // Allocate Memory
    csr.row_ptr = new int[num_nodes + 1];
    csr.col_ind = new int[num_edges];
    std::memset(csr.row_ptr, 0, sizeof(int) * (num_nodes + 1));

    // Compute histogram
    for (int i = 0; i < num_edges; ++i) {
        int src = edge_list[i].x;
        if (src < num_nodes) {
            csr.row_ptr[src + 1]++;
        }
    }

    // Prefix Sum
    for (int i = 0; i < num_nodes; ++i) {
        csr.row_ptr[i + 1] += csr.row_ptr[i];
    }

    // Fill col_ind
    int* current_offset = new int[num_nodes];
    
    // Initialize current_offset with the starting positions from row_ptr
    for(int i = 0; i < num_nodes; ++i) {
        current_offset[i] = csr.row_ptr[i];
    }

    for (int i = 0; i < num_edges; ++i) {
        int src = edge_list[i].x;
        int dest = edge_list[i].y;

        if (src < num_nodes) {
            // Place the destination in the correct spot
            int write_pos = current_offset[src];
            csr.col_ind[write_pos] = dest;

            // Increment the offset for this specific node so the next edge 
            // from this source goes into the next slot.
            current_offset[src]++;
        }
    }

    // Clean up temporary memory
    delete[] current_offset;

    return csr;
}


/**
 * Reads a 2SAT instance from a DIMACS CNF file and constructs its implication graph in CSR format.
 */

void read2SATInstance(const std::string& filename, int& num_vars, int& num_clauses, int& asp_result, CSRRepr& graph) {
    std::ifstream file(filename);
    if (!file) {
        std::cerr << "Error opening file: " << filename << std::endl;
        exit(EXIT_FAILURE);
    }

    std::string token;
    while (file >> token) {
        // read fixed variables
        // c fixed-timeout: 154
        if (token == "c") {
            if (file >> token && (token == "fixed-timeout:" || token == "fixed:")) {
                file >> asp_result;
            }
        }

        if (token == "p") {
            // p cnf num_vars num_clauses
            file >> token;
            if (token != "cnf") {
                std::cerr << "Unsupported format: " << token << std::endl;
                exit(EXIT_FAILURE);
            }
            file >> num_vars;
            file >> num_clauses;

            // Start reading clauses and build edge list
            int lit1, lit2, zero;
            std::vector<int2> edges;
            for (int i = 0; i < num_clauses; ++i) {
                file >> lit1 >> lit2 >> zero;
                // Add edges for implications
                int from1 = get_vertex_from_literal(-lit1);
                int to1   = get_vertex_from_literal(lit2);
                int from2 = get_vertex_from_literal(-lit2);
                int to2   = get_vertex_from_literal(lit1);

                edges.emplace_back(int2{from1, to1});
                edges.emplace_back(int2{from2, to2});
            }

            // Create CSR graph using the correct algorithm
            graph = createCSRGraph(num_vars * 2, edges.size(), edges.data());

            break;
        }
    }

}

#endif // COMMON_H


Writing /content/parallel-architectures/include/common.h


In [4]:
%%writefile /content/parallel-architectures/include/cuda_utils.h
#ifndef CUDA_UTILS_H
#define CUDA_UTILS_H

#include <cuda_runtime.h>
#include <iostream>

/**
 * CUDA Error Checking Macro
 * 
 * Usage:
 *   CUDA_CHECK(cudaMalloc(&d_ptr, size));
 *   CUDA_CHECK(cudaMemcpy(d_ptr, h_ptr, size, cudaMemcpyHostToDevice));
 * 
 * This macro will check the return value of any CUDA API call
 * and print an error message with file/line information if it fails.
 */
#define CUDA_CHECK(call) \
    do { \
        cudaError_t error = call; \
        if (error != cudaSuccess) { \
            std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ << " - " \
                      << cudaGetErrorString(error) << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while(0)

/*
#define CUSPARSE_CHECK(call) \
    do { \
        cusparseStatus_t status = call; \
        if (status != CUSPARSE_STATUS_SUCCESS) { \
            std::cerr << "cuSPARSE error at " << __FILE__ << ":" << __LINE__ << " - " \
                      << cusparseGetErrorString(status) << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while(0)
*/

/**
 * Print GPU device properties
 */
inline void printDeviceInfo(int device = 0) {
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, device));
    
    std::cout << "========================================" << std::endl;
    std::cout << "GPU Device Information:" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "Device name: " << prop.name << std::endl;
    std::cout << "Compute capability: " << prop.major << "." << prop.minor << std::endl;
    std::cout << "Global memory: " << prop.totalGlobalMem / (1024.0 * 1024.0) << " MB" << std::endl;
    std::cout << "Shared memory per block: " << prop.sharedMemPerBlock / 1024.0 << " KB" << std::endl;
    std::cout << "Registers per block: " << prop.regsPerBlock << std::endl;
    std::cout << "Warp size: " << prop.warpSize << std::endl;
    std::cout << "Max threads per block: " << prop.maxThreadsPerBlock << std::endl;
    std::cout << "Max threads dimensions: [" 
              << prop.maxThreadsDim[0] << ", "
              << prop.maxThreadsDim[1] << ", "
              << prop.maxThreadsDim[2] << "]" << std::endl;
    std::cout << "Max grid dimensions: [" 
              << prop.maxGridSize[0] << ", "
              << prop.maxGridSize[1] << ", "
              << prop.maxGridSize[2] << "]" << std::endl;
    std::cout << "Multiprocessor count: " << prop.multiProcessorCount << std::endl;
    std::cout << "Memory bus width: " << prop.memoryBusWidth << " bits" << std::endl;
    std::cout << "========================================" << std::endl;
}

// Utility function to free CSR graph memory in the device
void freeDeviceCSRRepr(CSRRepr& graph) {
    CUDA_CHECK(cudaFree(graph.row_ptr));
    CUDA_CHECK(cudaFree(graph.col_ind));

    graph.num_nodes = 0;
    graph.num_edges = 0;
    graph.row_ptr = nullptr;
    graph.col_ind = nullptr;
}

// Convert CSR to CSC format using cuSPARSE
// the input CSR graph is expected to be on the device and the output CSC graph will also be on the device
/*
CSRRepr CSR2CSC(const CSRRepr& d_csr) {
    int *d_csc_col_ptr, *d_csc_row_ind;

    CUDA_CHECK(cudaMalloc(&d_csc_col_ptr, (d_csr.num_nodes + 1) * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_csc_row_ind, d_csr.num_edges * sizeof(int)));


    // Set up cuSPARSE
    cusparseHandle_t handle;
    CUSPARSE_CHECK(cusparseCreate(&handle));

    size_t bufferSize = 0;
    void* d_buffer = nullptr;

    // Get buffer size
    CUSPARSE_CHECK(cusparseCsr2cscEx2_bufferSize(
        handle, d_csr.num_nodes, d_csr.num_nodes, d_csr.num_edges,
        nullptr, d_csr.row_ptr, d_csr.col_ind,
        nullptr, d_csc_col_ptr, d_csc_row_ind,
        CUDA_R_32F, CUSPARSE_ACTION_SYMBOLIC,
        CUSPARSE_INDEX_BASE_ZERO, CUSPARSE_CSR2CSC_ALG_DEFAULT, &bufferSize
    ));

    CUDA_CHECK(cudaMalloc(&d_buffer, bufferSize));

    // Perform transpose
    CUSPARSE_CHECK(cusparseCsr2cscEx2(
        handle, d_csr.num_nodes, d_csr.num_nodes, d_csr.num_edges,
        nullptr, d_csr.row_ptr, d_csr.col_ind,
        nullptr, d_csc_col_ptr, d_csc_row_ind,
        CUDA_R_32F, CUSPARSE_ACTION_SYMBOLIC,
        CUSPARSE_INDEX_BASE_ZERO, CUSPARSE_CSR2CSC_ALG_DEFAULT, d_buffer
    ));

    // Clean up
    CUSPARSE_CHECK(cusparseDestroy(handle));
    CUDA_CHECK(cudaFree(d_buffer));

    CSRRepr csc;
    csc.num_nodes = d_csr.num_nodes;
    csc.num_edges = d_csr.num_edges;
    csc.row_ptr = d_csc_col_ptr;
    csc.col_ind = d_csc_row_ind;

    return csc;
}
*/

/*
========================================
GPU Device Information - Colab
========================================
Device name: Tesla T4
Compute capability: 7.5
Global memory: 15095.1 MB
Shared memory per block: 48 KB
Registers per block: 65536
Warp size: 32
Max threads per block: 1024
Max threads dimensions: [1024, 1024, 64]
Max grid dimensions: [2147483647, 65535, 65535]
Multiprocessor count: 40
Memory clock rate: 5001 MHz
Memory bus width: 256 bits
========================================
*/

#endif // CUDA_UTILS_H

Writing /content/parallel-architectures/include/cuda_utils.h


In [14]:
%%writefile /content/parallel-architectures/src/SCC.cu
#include "../include/common.h"
#include "../include/cuda_utils.h"

#include <thrust/device_ptr.h>
#include <thrust/device_vector.h>
#include <thrust/scan.h>
#include <thrust/sort.h>
#include <thrust/transform.h>
#include <thrust/unique.h>

struct CondensedGraphResult {
    CSRRepr graph;
    int* d_scc_lookup;
};


// Initialize work list by adding all edges and io_max array with (v, max_outgoing_neighbor)
__global__ void globalInit(
    const CSRRepr g, 
    int2* const __restrict__ wl, 
    int2* const __restrict__ io_max
) {
    const int tid = threadIdx.x + blockIdx.x * NumThPerBlock;
    const int stride = gridDim.x * NumThPerBlock;

    for (int i = tid; i < g.num_nodes; i += stride) {
        const int begin = g.row_ptr[i];
        const int end   = g.row_ptr[i + 1];
        int y = i;
        // neighbors of i
        for (int j = begin; j < end; j++) {
            const int w = g.col_ind[j];
            wl[j] = int2{i, w};
            y = max(y, w);
        }
        io_max[i] = int2{i, y};
    }
}

// After edges are eliminated, re-initialize io_max to (i, i) for all nodes
// check if any changes were made and set go_again accordingly
__global__ void localInit(
    const int num_nodes, 
    int2* const __restrict__ io_max, 
    volatile bool* const __restrict__ go_again
) {
    const int tid = threadIdx.x + blockIdx.x * NumThPerBlock;
    const int stride = gridDim.x * NumThPerBlock;

    bool again = false;
    for (int i = tid; i < num_nodes; i += stride) {
        const int2 val = io_max[i];
        if (val.x != val.y) {
            io_max[i] = int2{i, i};
            again = true;
        }
    }

    // Check if any thread set again to true
    again = __syncthreads_or(again);
    // if not, computation is done
    if ((tid == 0) && again) {
        *go_again = true;
    }
}

// Propagate maximum signatures along the edges in the work list
// call this function until go_again is false
__global__ void propagateMax(
    const int2* const __restrict__ wl, 
    const int wl_size, 
    int2* const __restrict__ io_max, 
    volatile bool* const __restrict__ go_again
) {
    const int tid = threadIdx.x + blockIdx.x * NumThPerBlock;
    const int stride = gridDim.x * NumThPerBlock;

    bool updated, again = false;
    do {
        updated = false;
        
        for (int i = tid; i < wl_size; i += stride) {
            const int2 edge = wl[i];
            const int u = edge.x;
            const int v = edge.y;

            const int2 io_u = io_max[u];
            const int2 io_v = io_max[v];

            int im = io_u.x;
            int om = io_v.y;

            // Path compression

            // Since the signature are initialized with vertex id and can only increase,
            // io_max[im].x >= im and io_max[om].y >= om
            if (im > u) im = io_max[im].x;
            if (om > v) om = io_max[om].y;

            // Update io_max of u and v
            if (io_u.x < im) { io_max[u].x = im; updated = true;}
            if (io_u.y < om) { io_max[u].y = om; updated = true;}
            if (io_v.x < im) { io_max[v].x = im; updated = true;}
            if (io_v.y < om) { io_max[v].y = om; updated = true;}

            // Before overwriting a signature value s in a vertex v with a larger value t,
            // we can check that the signature value in s is less than t, in which case we update it to t.
            if ((io_u.x < om) && (io_max[io_u.x].y < om)) {io_max[io_u.x].y = om; updated = true;}
            if ((io_u.x != io_v.x) && (io_v.x < om) && (io_max[io_v.x].y < om)) {io_max[io_v.x].y = om; updated = true;}
            if ((io_u.y < im) && (io_max[io_u.y].x < im)) {io_max[io_u.y].x = im; updated = true;}
            if ((io_u.y != io_v.y) && (io_v.y < im) && (io_max[io_v.y].x < im)) {io_max[io_v.y].x = im; updated = true;}           

        }
        // If any updates are made we need to run another iteration
        again |= updated;

    } while (__syncthreads_or(updated));

    // If any thread made an update, set go_again to true
    again = __syncthreads_or(again);
    // if not, computation is done
    if ((threadIdx.x == 0) && again) {
        *go_again = true;
    }
}

// Remove edges that cannot be part of an SCC
// An edge (u,v) can be part of an SCC only if both endpoints have the same signature.
// Also edges leading to nodes already in an SCC are removed.
__global__ void removeEdges(
    const int2* const __restrict__ wl_in,
    int2* const __restrict__ wl_out, 
    const int wl_size, 
    int* const __restrict__ wl_out_size, 
    const int2* const __restrict__ io_max
) {
    const int tid = threadIdx.x + blockIdx.x * NumThPerBlock;
    const int stride = gridDim.x * NumThPerBlock;

    for (int i = tid; i < wl_size; i += stride) {
        const int2 edge = wl_in[i];
        const int u = edge.x;
        const int v = edge.y;

        const int2 io_u = io_max[u];
        const int2 io_v = io_max[v];

        // Keep edge only if both endpoints have the same signature and we are
        // not yet in an SCC (we are when io_v.x == io_v.y)
        if ((io_v.x != io_v.y) && (io_u.x == io_v.x) && (io_u.y == io_v.y)) {
            // atomic append to output work list
            const int pos = atomicAdd(wl_out_size, 1);
            wl_out[pos] = edge;
        }
    }
}

int* computeSCC(const CSRRepr& graph) {
    // work lists for edges
    int2 *d_wl1, *d_wl2;
    int wl_size = graph.num_edges;
    int *d_wl_size;
    // signature value for each node
    int2 *d_io_max;

    CUDA_CHECK(cudaMalloc(&d_wl1, graph.num_edges * sizeof(int2)));
    CUDA_CHECK(cudaMalloc(&d_wl2, graph.num_edges * sizeof(int2)));
    CUDA_CHECK(cudaMalloc(&d_wl_size, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_io_max, graph.num_nodes * sizeof(int2)));

    // initialize work lists and io_max
    int blocks = gridStrideBlocks(graph.num_nodes);
    globalInit<<<blocks, NumThPerBlock>>>(graph, d_wl1, d_io_max);

    bool go_again = true;
    bool *d_go_again;
    CUDA_CHECK(cudaMalloc(&d_go_again, sizeof(bool)));
    while (go_again) {
        // Propagate max values
        while (go_again) {
            CUDA_CHECK(cudaMemsetAsync(d_go_again, false, sizeof(bool))); // d_go_again = false;
            propagateMax<<<MaxBlocks, NumThPerBlock>>>(d_wl1, wl_size, d_io_max, d_go_again);
            // copy back go_again
            CUDA_CHECK(cudaMemcpy(&go_again, d_go_again, sizeof(bool), cudaMemcpyDeviceToHost));
        }

        // Remove edges that cannot be part of an SCC
        CUDA_CHECK(cudaMemsetAsync(d_wl_size, 0, sizeof(int))); // d_wl_size = 0;
        removeEdges<<<MaxBlocks, NumThPerBlock>>>(d_wl1, d_wl2, wl_size, d_wl_size, d_io_max);
        // New working list is in d_wl2, swap pointers
        std::swap(d_wl1, d_wl2);
        // copy back new wl_size
        CUDA_CHECK(cudaMemcpyAsync(&wl_size, d_wl_size, sizeof(int), cudaMemcpyDeviceToHost));

        // Local re-initialization
        CUDA_CHECK(cudaMemsetAsync(d_go_again, 0, sizeof(bool))); // d_go_again = false;
        localInit<<<MaxBlocks, NumThPerBlock>>>(graph.num_nodes, d_io_max, d_go_again);
        // copy back go_again
        CUDA_CHECK(cudaMemcpy(&go_again, d_go_again, sizeof(bool), cudaMemcpyDeviceToHost));
    }

    // Cleanup work lists and flags
    CUDA_CHECK(cudaFree(d_wl1));
    CUDA_CHECK(cudaFree(d_wl2));
    CUDA_CHECK(cudaFree(d_wl_size));
    CUDA_CHECK(cudaFree(d_go_again));

    // Convert io_max to just one signature value per node
    int *d_ssc_lookup;
    CUDA_CHECK(cudaMalloc(&d_ssc_lookup, graph.num_nodes * sizeof(int)));
    thrust::device_ptr<int2> dev_io_max_ptr(d_io_max);
    thrust::device_ptr<int> dev_id_map_ptr(d_ssc_lookup);
    thrust::transform(dev_io_max_ptr, dev_io_max_ptr + graph.num_nodes, dev_id_map_ptr, [] __device__ (const int2& val) {
        return val.x;
    });
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaFree(d_io_max));

    return d_ssc_lookup;
}

// Create a mapping from old SCC IDs to new contiguous SCC IDs, i.e.
// new_id[i] = id_map_out[old_id[i]]
__global__ void createMapping(
    const int* const __restrict__ unique_ids, 
    const int num_unique_ids, 
    int* const __restrict__ id_map_out
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_unique_ids) return;

    id_map_out[unique_ids[tid]] = tid;
}

// Remap SCC IDs in id_map_out using id_map_in
__global__ void mapSCCIds(
    const int* const __restrict__ id_map_in, 
    const int num_nodes, 
    int* const __restrict__ id_map_out
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_nodes) return;
    
    const int old_id = id_map_out[tid];
    id_map_out[tid] = id_map_in[old_id];
}

// Remap SCC IDs to contiguous range [0, num_sccs-1]
int remapSCCIds(int num_nodes, int* d_ssc_lookup) {
    // make it so id_map contains contiguous ids from 0 to num_scc-1
    thrust::device_ptr<int> dev_ssc_lookup_ptr(d_ssc_lookup);
    thrust::device_vector<int> d_unique_ids(dev_ssc_lookup_ptr, dev_ssc_lookup_ptr + num_nodes);
    thrust::sort(d_unique_ids.begin(), d_unique_ids.end());
    auto new_end = thrust::unique(d_unique_ids.begin(), d_unique_ids.end());
    const int h_scc_node_count = new_end - d_unique_ids.begin();
    
    int* d_id_map;
    CUDA_CHECK(cudaMalloc(&d_id_map, num_nodes * sizeof(int)));
    
    int blocks_mapping = (h_scc_node_count + NumThPerBlock - 1) / NumThPerBlock;
    createMapping<<<blocks_mapping, NumThPerBlock>>>(thrust::raw_pointer_cast(d_unique_ids.data()), h_scc_node_count, d_id_map);
    CUDA_CHECK(cudaDeviceSynchronize());
    // d_unique_ids goes out of scope and automatically frees its memory

    int blocks_remap = (num_nodes + NumThPerBlock - 1) / NumThPerBlock;
    mapSCCIds<<<blocks_remap, NumThPerBlock>>>(d_id_map, num_nodes, d_ssc_lookup);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaFree(d_id_map));
    
    return h_scc_node_count;
}

// Create edge list for condensed graph of SCCs
__global__ void createEdgeList(
    const CSRRepr g, 
    const int* const __restrict__ scc_lookup, 
    int2* const __restrict__ scc_edges, 
    int* const __restrict__ scc_edge_count
) {
    const int tid = threadIdx.x + blockIdx.x * NumThPerBlock;
    const int stride = gridDim.x * NumThPerBlock;

    for (int i = tid; i < g.num_nodes; i += stride) {
        const int begin = g.row_ptr[i];
        const int end   = g.row_ptr[i + 1];
        const int scc_u = scc_lookup[i];
        for (int j = begin; j < end; j++) {
            const int v = g.col_ind[j];
            const int scc_v = scc_lookup[v];
            if (scc_u != scc_v) {
                // atomic append to scc_edges
                const int pos = atomicAdd(scc_edge_count, 1);
                scc_edges[pos] = int2{scc_u, scc_v};
            }
        }
    }
}

// Count number of outgoing edges per node to build row_ptr
__global__ void countEdgesPerNode(
    const int2* const __restrict__ edges, 
    const int num_edges, 
    int* const __restrict__ row_ptr
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_edges) return;

    const int2 edge = edges[tid];
    atomicAdd(&row_ptr[edge.x + 1], 1);
}

// Build a graph in csr format from its edge list
CSRRepr buildCSRFromEdgeList(int2* d_edges, int num_edges, int num_nodes) {
    int* d_row_ptr;
    int* d_col_ind;
    CUDA_CHECK(cudaMalloc(&d_row_ptr, (num_nodes + 1) * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_col_ind, num_edges * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_row_ptr, 0, (num_nodes + 1) * sizeof(int)));

    // Sort edge list (required to make CSR rows contiguous)
    thrust::device_ptr<int2> dev_edges_ptr(d_edges);
    thrust::sort(dev_edges_ptr, dev_edges_ptr + num_edges, [] __device__ (const int2& a, const int2& b) {
        return a.x < b.x || (a.x == b.x && a.y < b.y);
    });
    CUDA_CHECK(cudaDeviceSynchronize());

    // Deduplicate identical edges (same src,dst)
    auto new_end = thrust::unique(dev_edges_ptr, dev_edges_ptr + num_edges,
        [] __device__ (const int2& a, const int2& b) {
            return a.x == b.x && a.y == b.y;
        }
    );
    num_edges = static_cast<int>(new_end - dev_edges_ptr);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Compute the histogram for row_ptr
    int blocks_histogram = (num_edges + NumThPerBlock - 1) / NumThPerBlock;
    countEdgesPerNode<<<blocks_histogram, NumThPerBlock>>>(d_edges, num_edges, d_row_ptr);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Compute prefix sum to get row_ptr
    thrust::device_ptr<int> dev_scc_row_ptr_ptr(d_row_ptr);
    thrust::inclusive_scan(dev_scc_row_ptr_ptr, dev_scc_row_ptr_ptr + num_nodes + 1, dev_scc_row_ptr_ptr);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Create col_ind by copying from edge list the second element of each edge
    thrust::device_ptr<int2> dev_scc_edges_ptr(d_edges);
    thrust::device_ptr<int> dev_scc_col_ind_ptr(d_col_ind);
    thrust::transform(dev_scc_edges_ptr, dev_scc_edges_ptr + num_edges, dev_scc_col_ind_ptr, [] __device__ (const int2& edge) {
        return edge.y;
    });
    CUDA_CHECK(cudaDeviceSynchronize());

    CSRRepr csr_graph;
    csr_graph.num_nodes = num_nodes;
    csr_graph.num_edges = num_edges;
    csr_graph.row_ptr = d_row_ptr;
    csr_graph.col_ind = d_col_ind;

    return csr_graph;
}

CondensedGraphResult computeCondensedGraph(const CSRRepr& graph) {
    int *d_ssc_lookup = computeSCC(graph);

    // Remap sparse SCC IDs to dense range [0, num_sccs-1]
    const int scc_node_count = remapSCCIds(graph.num_nodes, d_ssc_lookup);

    // Create new edge list for condensed graph
    int2* d_scc_edges;
    CUDA_CHECK(cudaMalloc(&d_scc_edges, graph.num_edges * sizeof(int2)));
    int* d_scc_edge_count;
    CUDA_CHECK(cudaMalloc(&d_scc_edge_count, sizeof(int)));
    CUDA_CHECK(cudaMemset(d_scc_edge_count, 0, sizeof(int)));

    int blocks = gridStrideBlocks(graph.num_nodes);
    createEdgeList<<<blocks, NumThPerBlock>>>(graph, d_ssc_lookup, d_scc_edges, d_scc_edge_count);
    CUDA_CHECK(cudaDeviceSynchronize());

    int h_scc_edge_count = 0;
    CUDA_CHECK(cudaMemcpy(&h_scc_edge_count, d_scc_edge_count, sizeof(int), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_scc_edge_count));

    // Build CSR representation of the condensed graph
    CSRRepr scc_graph = buildCSRFromEdgeList(d_scc_edges, h_scc_edge_count, scc_node_count);

    // Cleanup
    CUDA_CHECK(cudaFree(d_scc_edges));

    return CondensedGraphResult{scc_graph, d_ssc_lookup};
}

__global__ void findSccRepresentatives(const int* scc_ids, int* representatives, int num_nodes) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int node_idx = tid; node_idx < num_nodes; node_idx += stride) {
        int my_scc = scc_ids[node_idx];

        // Atomically set the representative for this SCC
        atomicExch(&representatives[my_scc], node_idx);
    }
}

// Given the mapping from original nodes to their SCC IDs, find one representative node for each SCC
void findRepresentatives(int* d_scc_ids, int num_lit, int* d_representatives) {
    int blocks = gridStrideBlocks(num_lit);
    findSccRepresentatives<<<blocks, NumThPerBlock>>>(d_scc_ids, d_representatives, num_lit);
}

Overwriting /content/parallel-architectures/src/SCC.cu


In [ ]:
%%writefile /content/parallel-architectures/src/topo_sort.cu
// Compute the topological sort of a DAG represented in CSR format
#include "../include/common.h"
#include "../include/cuda_utils.h"

#include <vector>

struct TopoResult {
    int* d_topo_order;
    std::vector<int> level_starts;
    int num_levels;
};

// Compute in-degrees of each node
__global__ void computeInDegrees(
    const int* const __restrict__ col_ind, 
    int* const __restrict__ in_degree, 
    int num_edges
) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = tid; i < num_edges; i += stride) {
        int target_node = col_ind[i];
        atomicAdd(&in_degree[target_node], 1);
    }
}

__global__ void findZeros(
    const int* const __restrict__ in_degree,
    int num_nodes,
    int* const __restrict__ queue,
    int* const __restrict__ queue_count
) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = tid; i < num_nodes; i += stride) {
        if (in_degree[i] == 0) {
            // Reserve a spot in the queue
            int idx = atomicAdd(queue_count, 1);
            queue[idx] = i;
        }
    }
}

__global__ void processFrontier(
    const int* const __restrict__ row_ptr, 
    const int* const __restrict__ col_ind, 
    int* const __restrict__ in_degree, 
    int* const __restrict__ topo_order,
    int* const __restrict__ global_counter,
    int current_level_start,
    int current_level_end
) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int idx = current_level_start + tid; idx < current_level_end; idx += stride) {
        // take element from input queue
        int u = topo_order[idx];
        
        // "Remove" the node by decreasing in-degrees of its neighbors
        for (int edge_idx = row_ptr[u]; edge_idx < row_ptr[u + 1]; edge_idx++) {
            int v = col_ind[edge_idx];
            
            int old_in_degree = atomicSub(&in_degree[v], 1);
            if (old_in_degree == 1) {
                // Add to output queue
                int out_idx = atomicAdd(global_counter, 1);
                topo_order[out_idx] = v;
            }
        }
    }
}

TopoResult topologicalSort(const CSRRepr& d_graph) {
    TopoResult result{};
    int num_nodes = d_graph.num_nodes;
    int num_edges = d_graph.num_edges;

    // Allocate device memory for topological order
    int* d_topo_order;
    CUDA_CHECK(cudaMalloc(&d_topo_order, num_nodes * sizeof(int)));

    // Counter for the back of the topo order queue
    int* d_counter;
    CUDA_CHECK(cudaMalloc(&d_counter, sizeof(int)));
    CUDA_CHECK(cudaMemset(d_counter, 0, sizeof(int)));

    result.d_topo_order = d_topo_order;

    // Prepare host memory for level starts
    result.level_starts.clear();
    result.level_starts.reserve(num_nodes + 1);

    // Allocate device memory for in-degrees and queue
    int* d_in_degree;
    CUDA_CHECK(cudaMalloc(&d_in_degree, num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_in_degree, 0, num_nodes * sizeof(int)));

    // Compute in-degrees
    int numBlocksEdges = gridStrideBlocks(num_edges);
    computeInDegrees<<<numBlocksEdges, NumThPerBlock>>>(d_graph.col_ind, d_in_degree, num_edges);

    // Find initial zero in-degree nodes
    int numBlocksNodes = gridStrideBlocks(num_nodes);
    findZeros<<<numBlocksNodes, NumThPerBlock>>>(d_in_degree, num_nodes, d_topo_order, d_counter);

    result.level_starts.push_back(0);

    while (true) {
        int processed_count;
        CUDA_CHECK(cudaMemcpy(&processed_count, d_counter, sizeof(int), cudaMemcpyDeviceToHost));

        int prev_level_start = result.level_starts.back();
        int current_level_count = processed_count - prev_level_start;

        if (current_level_count == 0) break; // No more nodes to process

        result.level_starts.push_back(processed_count);

        // Process current level
        int numBlocksLevel = gridStrideBlocks(current_level_count);
        processFrontier<<<numBlocksLevel, NumThPerBlock>>>(
            d_graph.row_ptr,
            d_graph.col_ind,
            d_in_degree,
            d_topo_order,
            d_counter,
            prev_level_start,
            processed_count
        );
    }

    // Clean up
    CUDA_CHECK(cudaFree(d_in_degree));
    CUDA_CHECK(cudaFree(d_counter));

    result.num_levels = result.level_starts.size() - 1;
    return result;
}

Writing /content/parallel-architectures/src/topo_sort.cu


In [7]:
%%writefile /content/parallel-architectures/src/back_bone.cu
#include "../include/common.h"
#include "../include/cuda_utils.h"

__global__ void dagSweep(
    const int* const __restrict__ row_ptr,
    const int* const __restrict__ col_ind,
    const int* const __restrict__ sorted_nodes,
    unsigned long long* const __restrict__ node_masks, // volatile??
    int* const __restrict__ assign_status,       // volatile??
    int level_start, int level_end,
    int batch_start_pair
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int idx = level_start + tid; idx < level_end; idx += stride) {
        int u = sorted_nodes[idx];

        // Bits propagated from parents
        unsigned long long my_mask = node_masks[u];

        int pair_id = u / 2;
        int bit_idx = pair_id - batch_start_pair;

        // u is in the batch
        if (bit_idx >= 0 && bit_idx < 64) {
            unsigned long long bit = (1ULL << bit_idx);

            // If bit is set, it means my "complement" is my ancestor. 
            // Since it's a DAG, I can't reach myself, so it must be them.
            if (my_mask & bit) {
                if (u % 2 == 0) {
                    assign_status[u]   = 1;
                    assign_status[u+1] = 0;
                } else {
                    assign_status[u]   = 0;
                    assign_status[u-1] = 1;
                }
            }

            // Next nodes are reachable by me
            my_mask |= bit;
            node_masks[u] = my_mask;
        }

        // Propagate to children
        if (my_mask != 0) {
            for (int i = row_ptr[u]; i < row_ptr[u+1]; i++) {
                int v = col_ind[i];
                atomicOr(&node_masks[v], my_mask);
            }
        }
    }
}


int* computeBackbone(
    const CSRRepr& d_graph,
    const TopoResult& topo_sort
) {
    int num_nodes = d_graph.num_nodes;
    // Bit i of d_node_masks[j] indicates whether the node j is reachable
    // from node 2*i or node 2*i+1
    unsigned long long* d_node_masks;
    // Status flags for original variables, 0 = FALSE, 1 = TRUE
    int* d_assign_status;
    CUDA_CHECK(cudaMalloc(&d_node_masks, d_graph.num_nodes * sizeof(unsigned long long)));
    CUDA_CHECK(cudaMalloc(&d_assign_status, d_graph.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_assign_status, -1, d_graph.num_nodes * sizeof(int)));

    // Loop over all pairs of literals in chunks of 64
    for (int batch = 0; batch < num_nodes/2; batch += 64) {
        
        // Reset masks to 0 for this batch
        cudaMemset(d_node_masks, 0, d_graph.num_nodes * sizeof(unsigned long long));

        // Propagate reachability information through the DAG in topological order
        for (size_t i = 0; i < topo_sort.num_levels; i++) {
            int start = topo_sort.level_starts[i];
            int end   = topo_sort.level_starts[i+1];
            int count = end - start;

            if (count == 0) continue;

            // Propagate reachability for this level
            int blocks = gridStrideBlocks(count);
            dagSweep<<<blocks, NumThPerBlock>>>(
                d_graph.row_ptr,
                d_graph.col_ind,
                topo_sort.d_topo_order,
                d_node_masks,
                d_assign_status,
                start,
                end,
                batch
            );
        }
    }

    // Cleanup
    CUDA_CHECK(cudaFree(d_node_masks));

    return d_assign_status;
}

Writing /content/parallel-architectures/src/back_bone.cu


In [8]:
%%writefile /content/parallel-architectures/src/WCC.cu
#include "../include/common.h"
#include "../include/cuda_utils.h"

#include <thrust/device_ptr.h>
#include <thrust/device_vector.h>
#include <thrust/iterator/constant_iterator.h>
#include <thrust/sequence.h>
#include <thrust/sort.h>

// Initialize each node to be its own parent
__global__ void initParent(
    int* const __restrict__ parent, 
    const int* const __restrict__ assign_status, 
    const int num_nodes
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = tid; i < num_nodes; i += stride) {
        if (assign_status && assign_status[i] != -1) {
            parent[i] = -1;
            continue;
        }
        parent[i] = i;
    }
}

/*
 * For each edge (u,v), attempt to hook the higher ID root to the lower ID root.
 */
__global__ void hook(
    const int* const __restrict__ row_ptr, 
    const int* const __restrict__ col_ind, 
    const int* const __restrict__ assign_status, 
    int* const __restrict__ parent, 
    volatile bool* const __restrict__ d_changed, 
    const int num_nodes
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int u = tid; u < num_nodes; u += stride) {
        if (assign_status && assign_status[u] != -1) {
            continue;
        }
        int start_edge = row_ptr[u];
        int end_edge = row_ptr[u + 1];

        int root_u = parent[u];

        for (int e = start_edge; e < end_edge; ++e) {
            int v = col_ind[e];
            if (assign_status && assign_status[v] != -1) {
                continue;
            }
            int root_v = parent[v];

            if (root_u != root_v) {
                // We want to attach the node with the Higher ID to the Lower ID
                int high = (root_u > root_v) ? root_u : root_v;
                int low  = (root_u > root_v) ? root_v : root_u;

                // Change the parent of the higher ID root to be the lower ID root
                int old = atomicMin(&parent[high], low);

                // If the parent actually changed, we must flag for another iteration
                if (old != low) {
                    *d_changed = true;
                    // Update root_u for potential subsequent edges
                    if (root_u == high) root_u = low; 
                }
            }
        }
    }
}

/*
 * Compress the trees by making each node point to its grandparent
 */
__global__ void compress(
    int* const __restrict__ parent, 
    const int* const __restrict__ assign_status, 
    const int num_nodes, 
    volatile bool* const __restrict__ d_changed
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = tid; i < num_nodes; i += stride) {
        if (assign_status && assign_status[i] != -1) {
            continue;
        }
        int p = parent[i];
        int gp = parent[p];

        if (p != gp) {
            parent[i] = gp;
            *d_changed = true;
        }
    }
}

/*
* Flatten the trees by making each node point to its grandparent until convergence
*/
__global__ void finalFlatten(
    int* const __restrict__ parent, 
    const int* const __restrict__ assign_status, 
    const int num_nodes
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = tid; i < num_nodes; i += stride) {
        if (assign_status && assign_status[i] != -1) {
            continue;
        }
        int p = parent[i];
        while (p != parent[p]) {
            p = parent[p];
        }
        parent[i] = p;
    }
}

int* computeWCC(CSRRepr& graph, const int* assign_status) {
    // Allocate device memory
    int* d_parent;
    bool* d_changed;
    int num_nodes = graph.num_nodes;
    CUDA_CHECK(cudaMalloc(&d_parent, num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_changed, sizeof(bool)));

    // Initialize Parents
    int blocks = gridStrideBlocks(num_nodes);
    initParent<<<blocks, NumThPerBlock>>>(d_parent, assign_status, num_nodes);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Hook and Compress until convergence
    bool h_changed = true;

    while (h_changed) {
        h_changed = false;
        CUDA_CHECK(cudaMemcpy(d_changed, &h_changed, sizeof(bool), cudaMemcpyHostToDevice));

        // Hook
        hook<<<blocks, NumThPerBlock>>>(graph.row_ptr, graph.col_ind, assign_status, d_parent, d_changed, num_nodes);
        // Compress
        compress<<<blocks, NumThPerBlock>>>(d_parent, assign_status, num_nodes, d_changed);
        
        // Check convergence flag
        CUDA_CHECK(cudaMemcpy(&h_changed, d_changed, sizeof(bool), cudaMemcpyDeviceToHost));
    }

    // Final Flattening
    finalFlatten<<<blocks, NumThPerBlock>>>(d_parent, assign_status, num_nodes);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Cleanup
    CUDA_CHECK(cudaFree(d_changed));

    return d_parent;
}

// Get CSR representation of the WCC mapping grouped by WCC ID, 
// i.e. all nodes in the same WCC are contiguous in col_ind and row_ptr indicates the start of each WCC group in col_ind
// NOTE: d_components_sorted_out should be preallocated with size num_nodes
CSRRepr getWCCGrouped(int* d_components_in, int* d_components_sorted_out, int num_nodes) {
    // Create a sequence [0, 1, 2...]
    int* d_col_idx;
    CUDA_CHECK(cudaMalloc(&d_col_idx, num_nodes * sizeof(int)));
    thrust::device_ptr<int> d_col_idx_ptr(d_col_idx);
    thrust::sequence(d_col_idx_ptr, d_col_idx_ptr + num_nodes);

    // Sort the component IDs and permute the node IDs accordingly
    CUDA_CHECK(cudaMemcpy(d_components_sorted_out, d_components_in, num_nodes * sizeof(int), cudaMemcpyDeviceToDevice));
    thrust::device_ptr<int> d_components_sorted(d_components_sorted_out);
    
    // Sort nodes based on component ID
    // d_components is sorted, d_col_idx is permuted to match
    thrust::sort_by_key(d_components_sorted, d_components_sorted + num_nodes, d_col_idx_ptr);

    // Allocate space for unique WCC IDs and their counts
    thrust::device_vector<int> d_unique_wcc_ids(num_nodes);
    thrust::device_vector<int> d_wcc_counts(num_nodes);

    // Count the number of nodes in each WCC
    auto end_it = thrust::reduce_by_key(
        d_components_sorted, 
        d_components_sorted + num_nodes, 
        thrust::constant_iterator<int>(1), // Each occurrence counts as 1
        d_unique_wcc_ids.begin(), 
        d_wcc_counts.begin()
    );

    int num_wccs = end_it.first - d_unique_wcc_ids.begin();

    // Compute the prefix sum
    int* d_row_ptr;
    CUDA_CHECK(cudaMalloc(&d_row_ptr, (num_wccs + 1) * sizeof(int)));
    thrust::device_ptr<int> d_row_ptr_ptr(d_row_ptr);
    thrust::exclusive_scan(d_wcc_counts.begin(), d_wcc_counts.begin() + num_wccs, d_row_ptr_ptr);
    // The last element of row_ptr should be the total number of nodes
    int h_last = num_nodes;
    CUDA_CHECK(cudaMemcpy(d_row_ptr + num_wccs, &h_last, sizeof(int), cudaMemcpyHostToDevice));

    CSRRepr wcc_grouped;
    wcc_grouped.num_nodes = num_wccs;
    wcc_grouped.num_edges = num_nodes;
    wcc_grouped.row_ptr = d_row_ptr;
    wcc_grouped.col_ind = d_col_idx;

    return wcc_grouped;
}


Writing /content/parallel-architectures/src/WCC.cu


In [9]:
%%writefile /content/parallel-architectures/src/heuristic.cu
#include "../include/common.h"
#include "../include/cuda_utils.h"

#include <bitset>

#include <thrust/device_ptr.h>
#include <thrust/execution_policy.h>
#include <thrust/functional.h>
#include <thrust/iterator/transform_iterator.h>
#include <thrust/iterator/zip_iterator.h>
#include <thrust/tuple.h>

// Assign false to the sinks in odd WCCs, and true to their complements
__global__ void assignSinks(
    const int* const __restrict__ row_ptr, 
    const int* const __restrict__ col_ind, 
    int num_nodes,
    const int* const __restrict__ wcc_map,
    int* const __restrict__ assignments
) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = tid; i < num_nodes; i += stride) {
        // Check if node is in an even WCC and not already assigned
        if (wcc_map[i] % 2 == 0 || assignments[i] != -1) {
            continue;
        }

        // Check all outgoing edges
        bool has_outgoing = false;
        for (int j = row_ptr[i]; j < row_ptr[i + 1]; j++) {
            int neighbor = col_ind[j];
            // If we find an outgoing edge to a node not assigned, then it's not a sink
            if (assignments[neighbor] == -1) {
                has_outgoing = true;
                break;
            }
        }

        // If no outgoing edges to unassigned nodes in odd WCCs, it's a sink
        if (!has_outgoing) {
            assignments[i] = 2;
            assignments[i ^ 1] = 3;
        }
    }
}

// Find sinks and put to them to false
void heuristic1(CSRRepr scc_graph, int* backbone_assignments, int* d_wcc_map) {
    int numBlocks = gridStrideBlocks(scc_graph.num_nodes);

    assignSinks<<<numBlocks, NumThPerBlock>>>(
        scc_graph.row_ptr, 
        scc_graph.col_ind, 
        scc_graph.num_nodes, 
        d_wcc_map, 
        backbone_assignments
    );
}


// Count the number of sinks for each WCC
__global__ void countSinks(
    const int* const __restrict__ row_ptr, 
    const int* const __restrict__ col_ind,
    const int* const __restrict__ assignments,
    int num_nodes,
    int* const __restrict__ num_sinks,
    bool* const __restrict__ is_sink,
    const int* const __restrict__ wcc_map
) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = tid; i < num_nodes; i += stride) {
        // Check if node is not already assigned
        if (assignments[i] != -1) {
            continue;
        }

        // Check all outgoing edges
        bool has_outgoing = false;
        for (int j = row_ptr[i]; j < row_ptr[i + 1]; j++) {
            int neighbor = col_ind[j];
            // If we find an outgoing edge to a node that is not assigned, then it's not a sink
            if (assignments[neighbor] == -1) {
                has_outgoing = true;
                break;
            }
        }

        // If no outgoing edges to unassigned nodes in WCCs, it's a sink
        if (!has_outgoing) {
            atomicAdd(&num_sinks[wcc_map[i]], 1);
            is_sink[i] = true;
        }
    }
}

// For each pair of nodes, check if one of them is a sink
// If so, assign false to the sink that has less sinks in its WCC, and true to the other one
__global__ void assignMinSinks(
    const int* const __restrict__ num_sinks,
    const bool* const __restrict__ is_sink,
    const int* const __restrict__ wcc_map,
    int num_pairs,
    int* const __restrict__ assignments
) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = tid; i < num_pairs; i += stride) {
        int idx0 = 2 * i;
        int idx1 = idx0 + 1;

        if (assignments[idx0] != -1) {
            continue; // Node already assigned
        }

        bool sink0 = is_sink[idx0];
        bool sink1 = is_sink[idx1];
        if (!sink0 && !sink1) {
            continue; // None of the nodes is a sink, skip
        }

        // Quello che è un sink, è nel WCC con meno sink?
        // Se si assegna false a quello, true all'altro. Altrimenti, lascia tutto com'è
        int wcc0 = wcc_map[idx0];
        int wcc1 = wcc_map[idx1];
        int n_sinks0 = num_sinks[wcc0];
        int n_sinks1 = num_sinks[wcc1];
        int has_wcc0_less_sinks = (n_sinks0 < n_sinks1) || (n_sinks0 == n_sinks1 && wcc0 < wcc1);
        
        if (sink0 && has_wcc0_less_sinks) {
            assignments[idx0] = 2; // False
            assignments[idx1] = 3; // True
        } else if (sink1 && !has_wcc0_less_sinks) {
            assignments[idx1] = 2; // False
            assignments[idx0] = 3; // True
        }
    }
}

// Find sources and sinks, check witch is less and assign accordingly
void heuristic2(CSRRepr scc_graph, int* backbone_assignments, int* d_wcc_map) {
    int* d_num_sinks;
    CUDA_CHECK(cudaMalloc(&d_num_sinks, scc_graph.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_num_sinks, 0, scc_graph.num_nodes * sizeof(int)));

    bool* d_is_sink;
    CUDA_CHECK(cudaMalloc(&d_is_sink, scc_graph.num_nodes * sizeof(bool)));
    CUDA_CHECK(cudaMemset(d_is_sink, false, scc_graph.num_nodes * sizeof(bool)));

    int numBlocks = gridStrideBlocks(scc_graph.num_nodes);
    countSinks<<<numBlocks, NumThPerBlock>>>(
        scc_graph.row_ptr,
        scc_graph.col_ind,
        backbone_assignments,
        scc_graph.num_nodes,
        d_num_sinks,
        d_is_sink,
        d_wcc_map
    );

    // Print d_num_sinks and d_is_sink for debugging
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int num_sinks;
        bool is_sink;
        CUDA_CHECK(cudaMemcpy(&num_sinks, d_num_sinks + i, sizeof(int), cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaMemcpy(&is_sink, d_is_sink + i, sizeof(bool), cudaMemcpyDeviceToHost));
        std::cout << "Node " << i << " has " << num_sinks << " sinks in its WCC and is_sink: " << is_sink << std::endl;
    }

    int num_pairs = scc_graph.num_nodes / 2;
    assignMinSinks<<<gridStrideBlocks(num_pairs), NumThPerBlock>>>(
        d_num_sinks,
        d_is_sink,
        d_wcc_map,
        num_pairs,
        backbone_assignments
    );

    // Print assignments for debugging
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int assignment;
        CUDA_CHECK(cudaMemcpy(&assignment, backbone_assignments + i, sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << "Node " << i << " assignment after heuristic2: " << assignment << std::endl;
    }

    // Cleanup
    CUDA_CHECK(cudaFree(d_num_sinks));
    CUDA_CHECK(cudaFree(d_is_sink));
}


// Propagate the number of nodes reachable forwards by each node in the DAG
// by processing the nodes in reverse topological order (children before parents)
__global__ void propagateReachability(
    int num_nodes,
    const int* const __restrict__ row_ptr,
    const int* const __restrict__ col_ind,
    const int* const __restrict__ nodes_in_level,      // Array of node IDs in this level
    int level_size,                 // How many nodes in this level
    const int* const __restrict__ assignments,
    unsigned long long* const __restrict__ reach_bits, // The bitset state (N x 1)
    int batch_start_idx             // The global index where this batch starts
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int idx = tid; idx < level_size; idx += stride) {
        int u = nodes_in_level[idx];

        // Node is already assigned, skip
        if (assignments[u] != -1) {
            reach_bits[u] = 0;
            continue;
        }

        // I can reach myself
        unsigned long long my_bits = 0;
        if (u >= batch_start_idx && u < batch_start_idx + 64) {
            my_bits = (1ULL << (u - batch_start_idx));
        }

        // I can reach all the nodes that my children can reach
        for (int e = row_ptr[u]; e < row_ptr[u+1]; ++e) {
            int v = col_ind[e];
            my_bits |= reach_bits[v];
        }

        reach_bits[u] = my_bits;
    }
}

// Count the number the number of nodes reachable in this batch
__global__ void accumulate_counts(
    int num_nodes, 
    const unsigned long long* const __restrict__ reach_bits, 
    int* const __restrict__ total_reach_counts
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int idx = tid; idx < num_nodes; idx += stride) {
        // __popcll counts number of set bits in a 64-bit integer
        total_reach_counts[idx] += __popcll(reach_bits[idx]);
    }
}

// Compute the number of nodes reachable by each node
// TODO: Maybe we can optime this by only computing the number reachability in each WCC
void countReachability(CSRRepr scc_graph, TopoResult topo_result, int* assignments, int* d_reach_counts) {
    unsigned long long *d_reach_bits;
    CUDA_CHECK(cudaMalloc(&d_reach_bits, scc_graph.num_nodes * sizeof(unsigned long long)));

    for (int batch_start = 0; batch_start < scc_graph.num_nodes; batch_start += 64) {
        // Initialize reachability bits for this batch
        CUDA_CHECK(cudaMemset(d_reach_bits, 0, scc_graph.num_nodes * sizeof(unsigned long long)));

        // Process levels in reverse topological order (children before parents)
        for (int level = topo_result.num_levels - 1; level >= 0; level--) {
            int level_start = topo_result.level_starts[level];
            int level_end = topo_result.level_starts[level + 1];
            int level_count = level_end - level_start;

            if (level_count == 0) continue;

            propagateReachability<<<gridStrideBlocks(level_count), NumThPerBlock>>>(
                scc_graph.num_nodes,
                scc_graph.row_ptr,
                scc_graph.col_ind,
                topo_result.d_topo_order + level_start,
                level_count,
                assignments,
                d_reach_bits,
                batch_start
            );

            // print the reachability bits for debugging
            for (int i = 0; i < scc_graph.num_nodes; i++) {
                unsigned long long bits;
                CUDA_CHECK(cudaMemcpy(&bits, d_reach_bits + i, sizeof(unsigned long long), cudaMemcpyDeviceToHost));
                std::cout << "Node " << i << " reachability bits: " << std::bitset<64>(bits) << std::endl;
            }
        }

        // After processing all levels, accumulate reachability counts
        accumulate_counts<<<gridStrideBlocks(scc_graph.num_nodes), NumThPerBlock>>>(
            scc_graph.num_nodes,
            d_reach_bits,
            d_reach_counts
        );
    }

    CUDA_CHECK(cudaFree(d_reach_bits));
}


// Propagate the deletion of a node in both "complement" DAGs, the target node is assigned true
__global__ void processFrontierDeletion(
    const int* const __restrict__ row_ptr, 
    const int* const __restrict__ col_ind, 
    int* const __restrict__ assignments, 
    const int* const __restrict__ queue_in,
    int* const __restrict__ queue_out,
    int* const __restrict__ out_count,
    int frontier_size
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int idx = tid; idx < frontier_size; idx += stride) {
        int u = queue_in[idx];

        // Propagate assignment to neighbors
        for (int i = row_ptr[u]; i < row_ptr[u+1]; ++i) {
            int v = col_ind[i];

            // If v in not assigned, assign it to true (and it complement to false) 
            // and add to next frontier
            if (atomicCAS(&assignments[v], -1, 1) == -1) {
                assignments[v ^ 1] = 0;
                int out_idx = atomicAdd(out_count, 1);
                queue_out[out_idx] = v;
            }
        }
    }
}

// For each WCC pair choose to set sinks or sources
__global__ void initTargetFrontier(
    const int* const __restrict__ max_nodes,
    const int* const __restrict__ max_counts,
    int num_pairs,
    int* const __restrict__ assignments,
    int* const __restrict__ queue_out,
    int* const __restrict__ out_count
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int p = tid; p < num_pairs; p += stride) {
        int dag_1 = 1 + (p * 2);
        int dag_2 = dag_1 + 1;
        int max_dag = (max_counts[dag_1] >= max_counts[dag_2]) ? dag_1 : dag_2;
        int target = max_nodes[max_dag];

        if (target == -1 || max_counts[max_dag] <= 0) {
            continue;
        }

        // Assign target and its complement, then enqueue target.
        if (atomicCAS(&assignments[target], -1, 3) == -1) {
            assignments[target ^ 1] = 2;
            int out_idx = atomicAdd(out_count, 1);
            queue_out[out_idx] = target;
        }
    }
}


// Find witch node in each pair of WCC reaches the most nodes, remove it 
// and all the nodes reachable by it and their complements accordingly
bool deleteAndPropagate(
    CSRRepr scc_graph,
    int* d_assignments,
    const int* d_max_nodes,
    const int* d_max_counts,
    int num_wcc_pairs
) {
    int *d_queue_in, *d_queue_out, *d_count_out;
    CUDA_CHECK(cudaMalloc(&d_queue_in, scc_graph.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_queue_out, scc_graph.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count_out, sizeof(int)));
    CUDA_CHECK(cudaMemset(d_count_out, 0, sizeof(int)));

    // Initialize the frontier with the target nodes to delete (assigned to true)
    initTargetFrontier<<<gridStrideBlocks(num_wcc_pairs), NumThPerBlock>>>(
        d_max_nodes,
        d_max_counts,
        num_wcc_pairs,
        d_assignments,
        d_queue_in,
        d_count_out
    );

    int queue_size = 0;
    CUDA_CHECK(cudaMemcpy(&queue_size, d_count_out, sizeof(int), cudaMemcpyDeviceToHost));
    bool has_targets = queue_size > 0;

    while (queue_size > 0) {
        CUDA_CHECK(cudaMemset(d_count_out, 0, sizeof(int)));

        // Propagate the deletion
        processFrontierDeletion<<<gridStrideBlocks(queue_size), NumThPerBlock>>>(
            scc_graph.row_ptr,
            scc_graph.col_ind,
            d_assignments,
            d_queue_in,
            d_queue_out,
            d_count_out,
            queue_size
        );

        CUDA_CHECK(cudaMemcpy(&queue_size, d_count_out, sizeof(int), cudaMemcpyDeviceToHost));
        std::swap(d_queue_in, d_queue_out);
    }

    CUDA_CHECK(cudaFree(d_queue_in));
    CUDA_CHECK(cudaFree(d_queue_out));
    CUDA_CHECK(cudaFree(d_count_out));

    return has_targets;
}


// Comparison of two pairs (count, node_id) to get the max count
struct MaxCountNode {
    __host__ __device__ thrust::tuple<int, int> operator()(
        const thrust::tuple<int, int>& a,
        const thrust::tuple<int, int>& b
    ) const {
        if (thrust::get<0>(a) > thrust::get<0>(b)) {
            return a;
        }
        return b;
    }
};

// Map node_id to its reachability count
struct CountLookup {
    const int* counts;
    __host__ __device__ int operator()(int node_id) const {
        return counts[node_id];
    }
};

// For each WCC find the node with the maximum reachability count and its count
void getMaxForWCC(
    const int* d_reach_counts, CSRRepr wcc_grouped, const int* d_sorted_wcc, // in
    int* d_wcc_max_nodes, int* d_wcc_max_counts                              // out
) {
    int* d_wcc_keys_out;
    CUDA_CHECK(cudaMalloc(&d_wcc_keys_out, wcc_grouped.num_nodes * sizeof(int)));

    CountLookup lookup{d_reach_counts};
    // Lazily map node_id -> reach_count without a temporary buffer.
    auto col_ind_ptr = thrust::device_ptr<int>(wcc_grouped.col_ind);
    auto counts_begin = thrust::make_transform_iterator(col_ind_ptr, lookup);
    // Values are (reach_count, node_id); reducer keeps the tuple with the larger count.
    auto values_begin = thrust::make_zip_iterator(
        thrust::make_tuple(
            counts_begin,
            col_ind_ptr
        )
    );
    auto out_values_begin = thrust::make_zip_iterator(
        thrust::make_tuple(
            thrust::device_ptr<int>(d_wcc_max_counts),
            thrust::device_ptr<int>(d_wcc_max_nodes)
        )
    );

    // WCC IDs must be grouped in d_sorted_wcc
    thrust::reduce_by_key(
        thrust::device,
        thrust::device_ptr<const int>(d_sorted_wcc),
        thrust::device_ptr<const int>(d_sorted_wcc + wcc_grouped.num_edges),
        values_begin,
        thrust::device_ptr<int>(d_wcc_keys_out),
        out_values_begin,
        cuda::std::equal_to<int>(),
        MaxCountNode()
    );

    CUDA_CHECK(cudaFree(d_wcc_keys_out));
}


// Compute the number of nodes reachable forwards or backwards. Find the node that reaches the most nodes, remove it and all the nodes ahead/behind accordingly, and repeat
void heuristic3(CSRRepr scc_graph, TopoResult topo_result, int* backbone_assignments, int* d_wcc_map) {
    // Get the reverse mapping
    int* d_sorted_wcc;
    CUDA_CHECK(cudaMalloc(&d_sorted_wcc, scc_graph.num_nodes * sizeof(int)));
    CSRRepr wcc_grouped = getWCCGrouped(d_wcc_map, d_sorted_wcc, scc_graph.num_nodes);

    // Count reachability for all nodes
    int* d_reach_counts;
    CUDA_CHECK(cudaMalloc(&d_reach_counts, scc_graph.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_reach_counts, 0, scc_graph.num_nodes * sizeof(int)));

    // Print assignments for debugging
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int assignment;
        CUDA_CHECK(cudaMemcpy(&assignment, backbone_assignments + i, sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << "Node " << i << " assignment: " << assignment << std::endl;
    }

    countReachability(scc_graph, topo_result, backbone_assignments, d_reach_counts);

    // print the reachability counts for debugging
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int count;
        CUDA_CHECK(cudaMemcpy(&count, d_reach_counts + i, sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << "Node " << i << " can reach " << count << " nodes." << std::endl;
    }

    // Allocate memory for the max node per WCC and its count
    int* d_max_nodes;
    int* d_max_counts;
    CUDA_CHECK(cudaMalloc(&d_max_nodes, wcc_grouped.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_max_counts, wcc_grouped.num_nodes * sizeof(int)));

    while (true) {
        // Get node with max reachability for each WCC
        getMaxForWCC(d_reach_counts, wcc_grouped, d_sorted_wcc, d_max_nodes, d_max_counts);

        // Print d_max_nodes for debugging
        std::cout << "Max reachability nodes for each WCC:" << std::endl;
        for (int i = 0; i < wcc_grouped.num_nodes; i++) {
            int max_node, max_count;
            CUDA_CHECK(cudaMemcpy(&max_node, d_max_nodes + i, sizeof(int), cudaMemcpyDeviceToHost));
            CUDA_CHECK(cudaMemcpy(&max_count, d_max_counts + i, sizeof(int), cudaMemcpyDeviceToHost));
            std::cout << "WCC " << i << " max node: " << max_node << " with count: " << max_count << std::endl;
        }

        // Delete all the nodes reachable by the node with max reachability in each WCC pair
        int num_wcc_pairs = (wcc_grouped.num_nodes - 1) / 2;
        bool h_has_change = deleteAndPropagate(
            scc_graph,
            backbone_assignments,
            d_max_nodes,
            d_max_counts,
            num_wcc_pairs
        );

        if (!h_has_change) {
            break; // No more nodes to remove
        }
        
        // Set d_reach_counts to 0 to all nodes that have backbone_assignments != -1
        thrust::device_ptr<int> d_reach_counts_ptr(d_reach_counts);
        thrust::device_ptr<int> d_assignments_ptr(backbone_assignments);
        thrust::transform(
            d_reach_counts_ptr,
            d_reach_counts_ptr + scc_graph.num_nodes,
            d_assignments_ptr,
            d_reach_counts_ptr,
            [] __device__ (int count, int assignment) {
                return assignment != -1 ? 0 : count;
            }
        );
    }

    // Cleanup
    CUDA_CHECK(cudaFree(d_sorted_wcc));
    CUDA_CHECK(cudaFree(d_reach_counts));
    CUDA_CHECK(cudaFree(d_max_nodes));
    CUDA_CHECK(cudaFree(d_max_counts));
}

// Same as heuristic3 but re-compute the reachability counts after each iteration.
void heuristic4(CSRRepr scc_graph, TopoResult topo_result, int* backbone_assignments, int* d_wcc_map) {
    // Get the reverse mapping
    int* d_sorted_wcc;
    CUDA_CHECK(cudaMalloc(&d_sorted_wcc, scc_graph.num_nodes * sizeof(int)));
    CSRRepr wcc_grouped = getWCCGrouped(d_wcc_map, d_sorted_wcc, scc_graph.num_nodes);

    // Count reachability for all nodes
    int* d_reach_counts;
    CUDA_CHECK(cudaMalloc(&d_reach_counts, scc_graph.num_nodes * sizeof(int)));

    // Allocate memory for the max node per WCC and its count
    int* d_max_nodes;
    int* d_max_counts;
    CUDA_CHECK(cudaMalloc(&d_max_nodes, wcc_grouped.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_max_counts, wcc_grouped.num_nodes * sizeof(int)));

    while (true) {
        // Count reachability for all nodes
        CUDA_CHECK(cudaMemset(d_reach_counts, 0, scc_graph.num_nodes * sizeof(int)));
        countReachability(scc_graph, topo_result, backbone_assignments, d_reach_counts);

        // Get node with max reachability for each WCC
        getMaxForWCC(d_reach_counts, wcc_grouped, d_sorted_wcc, d_max_nodes, d_max_counts);

        // Delete all the nodes reachable by the node with max reachability in each WCC pair
        int num_wcc_pairs = (wcc_grouped.num_nodes - 1) / 2;
        bool h_has_change = deleteAndPropagate(
            scc_graph,
            backbone_assignments,
            d_max_nodes,
            d_max_counts,
            num_wcc_pairs
        );

        if (!h_has_change) {
            break; // No more nodes to remove
        }
    }

    // Cleanup
    CUDA_CHECK(cudaFree(d_sorted_wcc));
    CUDA_CHECK(cudaFree(d_reach_counts));
    CUDA_CHECK(cudaFree(d_max_nodes));
    CUDA_CHECK(cudaFree(d_max_counts));
}


Writing /content/parallel-architectures/src/heuristic.cu


In [10]:
%%writefile /content/parallel-architectures/src/parallel.cu
#include "../include/common.h"
#include "../include/cuda_utils.h"
#include "SCC.cu"
#include "topo_sort.cu"
#include "back_bone.cu"
#include "WCC.cu"
#include "heuristic.cu"

std::vector<std::pair<int, int>> compute_assignments(int* d_assignments, int num_nodes);
void printAssignments(int* d_assignments, int* scc_map, int num_nodes, int num_lit, int heuristic);
void printNumAssignments(int* d_assignments, int* scc_map, int num_nodes, int num_lit, int heuristic);

int main(int argc, char* argv[]) {
    if (argc < 2) {
        std::cerr << "Usage: " << argv[0] << " <filename> [heuristic]" << std::endl;
        std::cerr << "  heuristic: 1 | 2 | 3 | 4 | all (default)" << std::endl;
        return 1;
    }

    std::string filename = argv[1];
    std::string heuristic = (argc >= 3) ? argv[2] : "all";
    
    // Print GPU information
    // printDeviceInfo();
    std::cout << std::endl;

    // read 2SAT instance from file
    int num_vars, num_clauses, asp_result;
    CSRRepr graph;
    read2SATInstance(filename, num_vars, num_clauses, asp_result, graph);

    // TODO: Start CUDA event timing

    // Allocate device memory
    CSRRepr d_graph;
    CUDA_CHECK(cudaMalloc(&d_graph.row_ptr, (graph.num_nodes + 1) * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_graph.col_ind, graph.num_edges * sizeof(int)));
    
    // Copy data from host to device
    CUDA_CHECK(cudaMemcpy(d_graph.row_ptr, graph.row_ptr, (graph.num_nodes + 1) * sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_graph.col_ind, graph.col_ind, graph.num_edges * sizeof(int), cudaMemcpyHostToDevice));
    d_graph.num_nodes = graph.num_nodes;
    d_graph.num_edges = graph.num_edges;

    // Compute the condensed graph of SCCs
    CondensedGraphResult condensed = computeCondensedGraph(d_graph);
    CSRRepr scc_graph = condensed.graph;

    // Print condensed graph info
    std::cout << "Condensed graph has " << scc_graph.num_nodes << " nodes and " << scc_graph.num_edges << " edges." << std::endl;
    // Print condensed.d_scc_lookup
    std::cout << "SCC Lookup Table: " << std::endl;
    for (int i = 0; i < graph.num_nodes; i++) {
        int scc_id;
        CUDA_CHECK(cudaMemcpy(&scc_id, &condensed.d_scc_lookup[i], sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << scc_id << " ";
    }
    std::cout << std::endl;
    // Print condensed graph
    std::cout << "Condensed Graph CSR Representation:" << std::endl;
    std::cout << "Row Ptr: " << std::endl;
    for (int i = 0; i <= scc_graph.num_nodes; i++) {
        int row_val;
        CUDA_CHECK(cudaMemcpy(&row_val, &scc_graph.row_ptr[i], sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << row_val << " ";
    }
    std::cout << std::endl;
    std::cout << "Col Ind: " << std::endl;
    for (int i = 0; i < scc_graph.num_edges; i++) {
        int col_val;
        CUDA_CHECK(cudaMemcpy(&col_val, &scc_graph.col_ind[i], sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << col_val << " ";
    }
    std::cout << std::endl;


    // Compute topological sort and levels for the condensed graph
    TopoResult topo_result = topologicalSort(scc_graph);

    // Print topological sort result
    std::cout << "Topological Sort Order: " << std::endl;
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int node;
        CUDA_CHECK(cudaMemcpy(&node, &topo_result.d_topo_order[i], sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << node << " ";
    }
    std::cout << std::endl;
    std::cout << "Number of levels in topological sort: " << topo_result.num_levels << std::endl;
    std::cout << "Level starts: " << std::endl;
    for (size_t i = 0; i < topo_result.level_starts.size(); i++) {
        std::cout << topo_result.level_starts[i] << " ";
    }
    std::cout << std::endl;

    int* d_backbone_assignments = computeBackbone(
        scc_graph,
        topo_result
    );

    // Print backbone assignments
    std::cout << "Backbone Assignments:" << std::endl;
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int assignment;
        CUDA_CHECK(cudaMemcpy(&assignment, &d_backbone_assignments[i], sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << assignment << " ";
    }
    std::cout << std::endl;

    // Calc WCCs
    int* d_wcc_map = computeWCC(scc_graph, d_backbone_assignments);

    // Print WCC
    std::cout << "WCC Map:" << std::endl;
    for (int i = 0; i < scc_graph.num_nodes; i++) {
        int wcc_id;
        CUDA_CHECK(cudaMemcpy(&wcc_id, &d_wcc_map[i], sizeof(int), cudaMemcpyDeviceToHost));
        std::cout << wcc_id << " ";
    }
    std::cout << std::endl;

    // Duplicate d_backbone_assignments to restore between heuristics
    int* d_backbone_assignments_base;
    CUDA_CHECK(cudaMalloc(&d_backbone_assignments_base, scc_graph.num_nodes * sizeof(int)));
    CUDA_CHECK(cudaMemcpy(
        d_backbone_assignments_base,
        d_backbone_assignments,
        scc_graph.num_nodes * sizeof(int),
        cudaMemcpyDeviceToDevice
    ));
    
    
    // 1. Find sinks and put to them to false
    if (heuristic == "1" || heuristic == "all") {
        CUDA_CHECK(cudaMemcpy(
            d_backbone_assignments,
            d_backbone_assignments_base,
            scc_graph.num_nodes * sizeof(int),
            cudaMemcpyDeviceToDevice
        ));
        heuristic1(scc_graph, d_backbone_assignments, d_wcc_map);
        printAssignments(d_backbone_assignments, condensed.d_scc_lookup, scc_graph.num_nodes, d_graph.num_nodes, 1);

    }
    // 2. Find sources and sinks, check witch is less and assign accordingly
    if (heuristic == "2" || heuristic == "all") {
        CUDA_CHECK(cudaMemcpy(
            d_backbone_assignments,
            d_backbone_assignments_base,
            scc_graph.num_nodes * sizeof(int),
            cudaMemcpyDeviceToDevice
        ));
        heuristic2(scc_graph, d_backbone_assignments, d_wcc_map);
        printAssignments(d_backbone_assignments, condensed.d_scc_lookup, scc_graph.num_nodes, d_graph.num_nodes, 2);
    }
    // 3. Compute the number of nodes reachable forwards or backwards. Find the node 
    // that reaches the most nodes, remove it and all the nodes ahead/behind accordingly, and repeat
    if (heuristic == "3" || heuristic == "all") {
        CUDA_CHECK(cudaMemcpy(
            d_backbone_assignments,
            d_backbone_assignments_base,
            scc_graph.num_nodes * sizeof(int),
            cudaMemcpyDeviceToDevice
        ));
        heuristic3(scc_graph, topo_result, d_backbone_assignments, d_wcc_map);
        printAssignments(d_backbone_assignments, condensed.d_scc_lookup, scc_graph.num_nodes, d_graph.num_nodes, 3);
    }
    // 4. Same as 3. but re-compute the reachability counts after each iteration.
    if (heuristic == "4" || heuristic == "all") {
        CUDA_CHECK(cudaMemcpy(
            d_backbone_assignments,
            d_backbone_assignments_base,
            scc_graph.num_nodes * sizeof(int),
            cudaMemcpyDeviceToDevice
        ));
        heuristic4(scc_graph, topo_result, d_backbone_assignments, d_wcc_map);
        printAssignments(d_backbone_assignments, condensed.d_scc_lookup, scc_graph.num_nodes, d_graph.num_nodes, 4);
    }
    if (heuristic != "1" && heuristic != "2" && heuristic != "3" && heuristic != "4" && heuristic != "all") {
        std::cerr << "Unknown heuristic: " << heuristic << std::endl;
        std::cerr << "Expected: 1 | 2 | 3 | 4 | all" << std::endl;
    }
    
    // - Free device memory
    freeDeviceCSRRepr(scc_graph);
    freeDeviceCSRRepr(d_graph);
    CUDA_CHECK(cudaFree(condensed.d_scc_lookup));
    CUDA_CHECK(cudaFree(topo_result.d_topo_order));
    CUDA_CHECK(cudaFree(d_backbone_assignments));
    CUDA_CHECK(cudaFree(d_backbone_assignments_base));
    CUDA_CHECK(cudaFree(d_wcc_map));
    // Free host memory
    freeCSRRepr(graph);
    
    // std::cout << "CUDA execution completed!" << std::endl;
    
    return 0;
}

// compute result from the final assignments array
std::vector<std::pair<int, int>> compute_assignments(int* d_assignments, int* scc_map, int num_nodes, int num_lit) {
    int* d_representatives;
    CUDA_CHECK(cudaMalloc(&d_representatives, num_nodes * sizeof(int)));
    // Get the representative for each node
    findRepresentatives(scc_map, num_lit, d_representatives);
    int* h_representatives = new int[num_nodes];
    int* h_assignments = new int[num_nodes];
    CUDA_CHECK(cudaMemcpy(h_representatives, d_representatives, num_nodes * sizeof(int), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_assignments, d_assignments, num_nodes * sizeof(int), cudaMemcpyDeviceToHost));

    std::vector<std::pair<int, int>> var_assignments;

    // Print h_rappresentatives  for debugging
    for (int i = 0; i < num_nodes; i++) {
        std::cout << "Node " << i << ": representative = " << h_representatives[i] << ", assignment = " << h_assignments[i] << std::endl;
    }

    for (int i = 0; i < num_nodes; i++) {
        if (h_assignments[i] == 3) {
            int var = h_representatives[i] / 2;
            if (h_representatives[i] % 2 == 0) {
                var_assignments.emplace_back(var, 1);
            } else {
                var_assignments.emplace_back(var, 0);
            }
        }
    }

    CUDA_CHECK(cudaFree(d_representatives));
    free(h_representatives);
    free(h_assignments);

    return var_assignments;
}

void printAssignments(int* d_assignments, int* scc_map, int num_nodes, int num_lit, int heuristic) {
    auto var_assignments = compute_assignments(d_assignments, scc_map, num_nodes, num_lit);
    std::cout << "h" << heuristic << ": " << var_assignments.size() << std::endl;
    for (const auto& [var, value] : var_assignments) {
        std::cout << var << value << std::endl;
    }
    std::cout << std::endl;
}

void printNumAssignments(int* d_assignments, int* scc_map, int num_nodes, int num_lit, int heuristic) {
    auto var_assignments = compute_assignments(d_assignments, scc_map, num_nodes, num_lit);
    std::cout << "h" << heuristic << ": " << var_assignments.size() << std::endl;
}


Writing /content/parallel-architectures/src/parallel.cu


In [ ]:
%%writefile /content/parallel-architectures/test_instance.cnf
p cnf 10 14
1 -6 0
4 -8 0
-2 7 0
-9 10 0
-8 3 0
3 -1 0
-4 7 0
-3 -9 0
5 9 0
-4 -1 0
-3 6 0
6 -8 0
-9 -5 0
-8 -5 0

Writing /content/parallel-architectures/test_instance.cnf


In [15]:
%%shell
cd /content/parallel-architectures
nvcc -std=c++17 -O3 -arch=sm_70 --extended-lambda -Iinclude src/parallel.cu -o parallel
./parallel test_instance.cnf

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

Condensed graph has 14 nodes and 16 edges.
SCC Lookup Table: 
4 5 0 1 4 5 2 3 11 10 4 5 6 7 8 9 10 11 12 13 
Condensed Graph CSR Representation:
Row Ptr: 
0 1 1 3 4 6 7 7 9 12 12 14 15 15 16 
Col Ind: 
6 5 6 9 3 11 9 1 3 2 4 10 5 12 9 11 
Topological Sort Order: 
0 7 8 13 1 2 4 10 3 5 6 11 12 9 
Number of levels in topological sort: 4
Level starts: 
0 4 8 13 14 
Backbone Assignments:
-1 -1 -1 -1 -1 -1 -1 -1 1 0 -1 -1 -1 -1 
WCC Map:
0 1 0 1 1 0 0 1 -1 -1 0 1 0 1 
Node 0: representative = 2, assignment = 3
Node 1: representative = 3, assignment = 2
Node 2: representative = 6, assignment = 3
Node 3: representative = 7, assignment = 2
Node 4: representative = 10, assignment = -1
Node 5: representative = 11, assignment = -1
Node 6: representative = 12, assignment = -1
Node 7: representative = 13, assignme

In [ ]:
%%shell
cd /content/parallel-architectures
./parallel test_instance.cnf


Condensed graph has 14 nodes and 16 edges.
SCC Lookup Table: 
4 5 0 1 4 5 2 3 11 10 4 5 6 7 8 9 10 11 12 13 
Condensed Graph CSR Representation:
Row Ptr: 
0 1 1 3 4 6 7 7 9 12 12 14 15 15 16 
Col Ind: 
6 5 6 9 3 11 9 1 3 2 4 10 5 12 9 11 
Topological Sort Order: 
0 7 8 13 1 2 4 10 3 5 6 11 12 9 
Number of levels in topological sort: 4
Level starts: 
0 4 8 13 14 
Backbone Assignments:
-1 -1 -1 -1 -1 -1 -1 -1 1 0 -1 -1 -1 -1 
WCC Map:
0 1 0 1 1 0 0 1 -1 -1 0 1 0 1 
h1: 3
11
31
40

Node 0 has 3 sinks in its WCC and is_sink: 0
Node 1 has 3 sinks in its WCC and is_sink: 1
Node 2 has 0 sinks in its WCC and is_sink: 0
Node 3 has 0 sinks in its WCC and is_sink: 1
Node 4 has 0 sinks in its WCC and is_sink: 0
Node 5 has 0 sinks in its WCC and is_sink: 1
Node 6 has 0 sinks in its WCC and is_sink: 1
Node 7 has 0 sinks in its WCC and is_sink: 0
Node 8 has 0 sinks in its WCC and is_sink: 0
Node 9 has 0 sinks in its WCC and is_sink: 0
Node 10 has 0 sinks in its WCC and is_sink: 0
Node 11 has 0 sinks